# Smallest stiffness eigenvalue versus strain — SIM 5000s, 5010s, 5100s, and 5110s

This notebook analyzes SIM 5000–5009, 5010–5019, 5100–5109, and 5110–5119. It prefers local results and then follows the plain-text `I001_Results/AAA_fwd` redirect for moved results. Eigenvalues come from the compact `DATA_PICK_*_EIG.json` files; the multi-GB `EIGV.pkl` eigenvector payloads are not needed.

Step 1 applies a 5 mm displacement over a 20 mm specimen dimension, so nominal compressive strain is `0.25 * t`. Complete histories contain 401 snapshots. Available partial histories are plotted and clearly reported instead of being discarded.

In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path('../I001_Results')
if not RESULTS_DIR.exists():
    RESULTS_DIR = Path('I001_Results')

RESULTS_DIRS = [RESULTS_DIR]
fwd_file = RESULTS_DIR / 'AAA_fwd'
if fwd_file.is_file():
    forwarded_dir = Path(fwd_file.read_text().strip()).expanduser()
    if forwarded_dir.is_dir() and forwarded_dir.resolve() != RESULTS_DIR.resolve():
        RESULTS_DIRS.append(forwarded_dir)

def resolve_result(filename):
    """Prefer a local result, then use the directory named by AAA_fwd."""
    return next((directory / filename for directory in RESULTS_DIRS
                 if (directory / filename).is_file()), None)

PRESSURES = (0.0,) + tuple(0.04 * np.sqrt(2.0) ** i for i in range(9))
FAMILIES = {
    'SIM 5000 family — hexagonal packing, linear material': range(5000, 5010),
    'SIM 5010 family — random packing, linear material': range(5010, 5020),
    'SIM 5100 family — hexagonal packing, neo-Hookean material': range(5100, 5110),
    'SIM 5110 family — random packing, neo-Hookean material': range(5110, 5120),
}
EXPECTED_EIGENVALUE_POINTS = 401
SPECIMEN_DIMENSION = 20.0  # mm
IMPOSED_DISPLACEMENT = 5.0  # mm
MAX_STRAIN = IMPOSED_DISPLACEMENT / SPECIMEN_DIMENSION

print('Result search order:')
for directory in RESULTS_DIRS:
    print(f'  {directory.resolve()}')
print(f'Nominal maximum compressive strain: {MAX_STRAIN:.1%}')


In [ ]:
curves = {}
missing = []
partial = []
invalid = []
missing_f = []

for family_name, sim_ids in FAMILIES.items():
    curves[family_name] = {}
    for sim_id in sim_ids:
        path = resolve_result(f'DATA_PICK_{sim_id}_EIG.json')
        if path is None:
            missing.append(sim_id)
            continue

        try:
            with path.open() as handle:
                snapshots = sorted(json.load(handle), key=lambda item: item['time'])
            time = np.asarray([item['time'] for item in snapshots], dtype=float)
            eigenvalues = np.asarray([item['eigenvalues'] for item in snapshots], dtype=float)
            if len(time) < 2:
                raise ValueError('fewer than two eigenvalue snapshots')
            if eigenvalues.ndim != 2 or eigenvalues.shape[1] == 0:
                raise ValueError('eigenvalues does not contain mode columns')
            if len(time) != len(eigenvalues):
                raise ValueError('time and eigenvalue lengths differ')
            if not np.all(np.diff(time) > 0.0):
                raise ValueError('snapshot times are not strictly increasing')
        except (KeyError, TypeError, ValueError, json.JSONDecodeError) as error:
            invalid.append((sim_id, str(error), str(path)))
            continue

        is_complete = len(time) == EXPECTED_EIGENVALUE_POINTS and np.isclose(time[-1], 1.0)
        if not is_complete:
            partial.append((sim_id, len(time), time[-1]))

        lambda_min = eigenvalues[:, 0]
        strain = MAX_STRAIN * time
        pressure = PRESSURES[sim_id % 10]
        crossing = np.flatnonzero((lambda_min[:-1] > 0.0) & (lambda_min[1:] <= 0.0))
        if len(crossing):
            i = int(crossing[0])
            t0, t1 = time[i:i + 2]
            l0, l1 = lambda_min[i:i + 2]
            crossing_t = t0 - l0 * (t1 - t0) / (l1 - l0)
        else:
            crossing_t = np.nan

        f_at_crossing = np.nan
        i3_path = resolve_result(f'DATA_PICK_{sim_id}_I3_BFS_3002.pkl')
        if i3_path is not None and np.isfinite(crossing_t):
            with i3_path.open('rb') as handle:
                i3 = pickle.load(handle)
            i3_time = np.asarray(i3['t'], dtype=float)
            ef_t = np.asarray(i3['global_ef_t'], dtype=float)
            ef_c = np.asarray(i3['global_ef_c'], dtype=float)
            if (len(i3_time) == len(ef_t) == len(ef_c) and len(i3_time) > 1
                    and i3_time[0] <= crossing_t <= i3_time[-1]):
                ef_t_crossing = np.interp(crossing_t, i3_time, ef_t)
                ef_c_crossing = np.interp(crossing_t, i3_time, ef_c)
                if ef_c_crossing != 0.0:
                    f_at_crossing = ef_t_crossing / ef_c_crossing
        if np.isfinite(crossing_t) and not np.isfinite(f_at_crossing):
            missing_f.append(sim_id)

        curves[family_name][sim_id] = {
            'time': time, 'strain': strain, 'lambda_min': lambda_min.copy(),
            'pressure': pressure, 'crossing_t': crossing_t,
            'f_at_crossing': f_at_crossing, 'complete': is_complete,
            'source': path,
        }

print('Loaded curves:')
for family_name, family_curves in curves.items():
    local_ids = [sim for sim, data in family_curves.items() if data['source'].parent == RESULTS_DIR]
    fwd_ids = [sim for sim, data in family_curves.items() if data['source'].parent != RESULTS_DIR]
    print(f'  {family_name}: {len(family_curves)}/10 (local={local_ids}, forwarded={fwd_ids})')
if missing:
    print(f'Missing eigenvalue histories: {missing}')
if partial:
    print('Partial histories retained (SIM, snapshots, final time):')
    for row in partial:
        print(f'  {row[0]}: {row[1]} points through t={row[2]:.4f}')
if invalid:
    print(f'Invalid eigenvalue histories: {invalid}')
if missing_f:
    print(f'f unavailable at an observed zero crossing for: {missing_f}')


In [ ]:
figures = {}
summary_figures = {}
summary_rows = {}

for family_name, family_curves in curves.items():
    family_id = min(FAMILIES[family_name])
    fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
    for sim_id, data in sorted(family_curves.items()):
        linestyle = '-' if data['complete'] else '--'
        label = f"SIM {sim_id}, p = {data['pressure']:.5g}"
        if not data['complete']:
            label += f" (partial: {len(data['time'])} points)"
        line, = ax.plot(100 * data['strain'], data['lambda_min'], linestyle=linestyle,
                        linewidth=1.5, label=label)
        if np.isfinite(data['crossing_t']):
            ax.scatter(100 * MAX_STRAIN * data['crossing_t'], 0.0,
                       color=line.get_color(), edgecolor='black', s=42, zorder=5)
    ax.axhline(0.0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Nominal compressive strain (%)')
    ax.set_ylabel('Smallest tangent-stiffness eigenvalue')
    ax.set_title(f'{family_name}: smallest eigenvalue versus strain')
    ax.grid(True, alpha=0.3)
    if family_curves:
        ax.legend(title='Simulation / internal pressure', fontsize=8, loc='best')
    figures[family_name] = fig
    plt.show()

    crossing_rows = np.asarray([
        (sim_id, data['pressure'], data['crossing_t'])
        for sim_id, data in sorted(family_curves.items())
        if np.isfinite(data['crossing_t'])
    ], dtype=float).reshape(-1, 3)
    f_rows = np.asarray([
        (sim_id, data['pressure'], data['crossing_t'], data['f_at_crossing'])
        for sim_id, data in sorted(family_curves.items())
        if np.isfinite(data['crossing_t']) and np.isfinite(data['f_at_crossing'])
    ], dtype=float).reshape(-1, 4)
    summary_rows[family_id] = {'crossing': crossing_rows, 'f': f_rows}
    summary_figures[family_id] = {}

    if len(crossing_rows):
        print(f'First eigenvalue zero crossings for SIM {family_id}s:')
        print('  SIM       pressure   compression ratio')
        for sim_id, pressure, compression_ratio in crossing_rows:
            print(f'  {int(sim_id):4d}   {pressure:10.6g}   {compression_ratio:17.8f}')
        fig_crossing, ax_crossing = plt.subplots(figsize=(9, 6), constrained_layout=True)
        ax_crossing.plot(crossing_rows[:, 1], crossing_rows[:, 2], 'o-', linewidth=1.5)
        for sim_id, pressure, compression_ratio in crossing_rows:
            ax_crossing.annotate(f'SIM {int(sim_id)}', (pressure, compression_ratio),
                                   xytext=(5, 5), textcoords='offset points', fontsize=8)
        ax_crossing.set_xlabel('Internal pressure, p')
        ax_crossing.set_ylabel('Compression ratio at first eigenvalue zero crossing')
        ax_crossing.set_title(f'SIM {family_id}s: first eigenvalue zero crossing')
        ax_crossing.grid(True, alpha=0.3)
        summary_figures[family_id]['crossing'] = fig_crossing
        plt.show()
    else:
        print(f'No zero crossing was found for the available SIM {family_id}s curves.')

    if len(f_rows):
        print(f'f at first eigenvalue zero crossing for SIM {family_id}s:')
        print('  SIM       pressure   compression ratio       f')
        for sim_id, pressure, compression_ratio, f_value in f_rows:
            print(f'  {int(sim_id):4d}   {pressure:10.6g}   {compression_ratio:17.8f}   {f_value: .8f}')
        for kind, x_column, xlabel, title_tail in [
            ('f_pressure', 1, 'Internal pressure, p', 'vs pressure'),
            ('f_ratio', 2, 'Compression ratio at first eigenvalue zero crossing',
             'vs compression ratio'),
        ]:
            fig_f, ax_f = plt.subplots(figsize=(9, 6), constrained_layout=True)
            ax_f.plot(f_rows[:, x_column], f_rows[:, 3], 'o-', linewidth=1.5)
            for row in f_rows:
                ax_f.annotate(f'SIM {int(row[0])}', (row[x_column], row[3]),
                              xytext=(5, 5), textcoords='offset points', fontsize=8)
            ax_f.set_xlabel(xlabel)
            ax_f.set_ylabel('f = average global tension efficiency / compression efficiency')
            ax_f.set_title(f'SIM {family_id}s: f at eigenvalue zero {title_tail}')
            ax_f.grid(True, alpha=0.3)
            summary_figures[family_id][kind] = fig_f
            plt.show()
    else:
        print(f'No valid f values were found at SIM {family_id}s zero crossings.')


In [ ]:
# Save all generated figures next to the notebook.
for family_name, figure in figures.items():
    family_id = min(FAMILIES[family_name])
    output_path = Path(f'SIM_{family_id}s_smallest_eigenvalue_vs_strain.png')
    figure.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'Saved: {output_path.resolve()}')

for family_id, family_figures in summary_figures.items():
    names = {
        'crossing': f'SIM_{family_id}s_first_eigenvalue_zero_crossing_vs_pressure.png',
        'f_pressure': f'SIM_{family_id}s_f_at_eigenvalue_zero_vs_pressure.png',
        'f_ratio': f'SIM_{family_id}s_f_at_eigenvalue_zero_vs_compression_ratio.png',
    }
    for figure_kind, figure in family_figures.items():
        output_path = Path(names[figure_kind])
        figure.savefig(output_path, dpi=200, bbox_inches='tight')
        print(f'Saved: {output_path.resolve()}')
